# Ejercicio 3 — Invariancia de la regresión lineal ante reescalamiento

**Introducción al Machine Learning — Cuarta lista de ejercicios — Daniel Vargas**

Se usa `experimento_3.csv` (columnas `x1`, `x2`, `y`). Se ajusta
$y=\beta_0+\beta_1 x_1+\beta_2 x_2$ y, con $z_2=x_2/100$, se ajusta de nuevo
$y=\gamma_0+\gamma_1 x_1+\gamma_2 z_2$.

In [1]:
import numpy as np
import pandas as pd

from lista4.data import cargar_csv, obtener_xy
from lista4.models import ajustar_regresion_lineal
from lista4.metrics import riesgo_cuadratico

df3 = cargar_csv('experimento_3.csv')
df3.head()

,x1,x2,y
0,-0.793122,205.811529,6.722415
1,0.240571,70.612604,5.569864
2,-1.896326,-109.601177,-14.546028
3,1.395772,-113.127226,2.012899
4,0.638295,-95.273569,-2.743663


## (a) Coeficientes y riesgos de los dos modelos

In [2]:
X1, y = obtener_xy(df3, ['x1', 'x2'], 'y')

df3_z = df3.copy()
df3_z['z2'] = df3_z['x2'] / 100.0
X2, _ = obtener_xy(df3_z, ['x1', 'z2'], 'y')

modelo_beta = ajustar_regresion_lineal(X1, y)
modelo_gamma = ajustar_regresion_lineal(X2, y)

R_beta = riesgo_cuadratico(y, modelo_beta.predict(X1))
R_gamma = riesgo_cuadratico(y, modelo_gamma.predict(X2))

beta0, beta1, beta2 = modelo_beta.intercept_, *modelo_beta.coef_
gamma0, gamma1, gamma2 = modelo_gamma.intercept_, *modelo_gamma.coef_

print('beta0, beta1, beta2 =', beta0, beta1, beta2, ' | R_beta =', R_beta)
print('gamma0, gamma1, gamma2 =', gamma0, gamma1, gamma2, ' | R_gamma =', R_gamma)

beta0, beta1, beta2 = -0.1014700964075996 5.012065732244869 0.050476925444612616  | R_beta = 0.9232252621022038
gamma0, gamma1, gamma2 = -0.10147009640759957 5.01206573224487 5.047692544461261  | R_gamma = 0.9232252621022037


## (b) Diferencia máxima entre las predicciones de los dos modelos

In [3]:
y_hat_1 = modelo_beta.predict(X1)
y_hat_2 = modelo_gamma.predict(X2)

d = np.abs(y_hat_1 - y_hat_2)
max_d = d.max()
print('max_i d_i =', max_d)

max_i d_i = 5.384581669432009e-15


## (c) Verificación numérica de la relación entre $\hat\beta$ y $\hat\gamma$

In [4]:
error_intercepto = abs(gamma0 - beta0)
error_beta1 = abs(gamma1 - beta1)
error_beta2 = abs(gamma2 - 100 * beta2)

print(f'|gamma0 - beta0|        = {error_intercepto:.3e}')
print(f'|gamma1 - beta1|        = {error_beta1:.3e}')
print(f'|gamma2 - 100*beta2|    = {error_beta2:.3e}')

|gamma0 - beta0|        = 2.776e-17
|gamma1 - beta1|        = 8.882e-16
|gamma2 - 100*beta2|    = 8.882e-16


## (d) Comparación de $|\hat\beta_2|$ y $|\hat\gamma_2|$, y de las predicciones

In [5]:
print(f'|beta2_hat|  = {abs(beta2):.6f}')
print(f'|gamma2_hat| = {abs(gamma2):.6f}')
print(f'|gamma2_hat| / |beta2_hat| = {abs(gamma2)/abs(beta2):.2f}  (se espera ~100)')
print()
print(f'max_i d_i = {max_d:.3e}  (diferencia maxima entre predicciones)')

|beta2_hat|  = 0.050477
|gamma2_hat| = 5.047693
|gamma2_hat| / |beta2_hat| = 100.00  (se espera ~100)

max_i d_i = 5.385e-15  (diferencia maxima entre predicciones)


**Respuesta.** Al dividir $x_2$ entre 100 para obtener $z_2$, el coeficiente
asociado se multiplica por 100 ($\hat\gamma_2 \approx 100\,\hat\beta_2$), de modo
que $|\hat\gamma_2|$ es unas 100 veces más grande que $|\hat\beta_2|$: el
reescalamiento cambia la *magnitud* del coeficiente, pero no cambia el
modelo en sí. Esto se confirma con que `max_i d_i` es prácticamente cero
(del orden del error numérico de la optimización): ambos modelos producen
exactamente las mismas predicciones, solo que repartidas de forma distinta
entre el coeficiente y la escala de la variable.

## (e) Tabla resumen

In [6]:
tabla_3 = pd.DataFrame([
    {'modelo': 'beta (x2 original)', 'intercepto': beta0, 'coef_x1': beta1,
     'coef_x2_o_z2': beta2, 'riesgo_empirico': R_beta, 'max_i d_i': max_d},
    {'modelo': 'gamma (z2 = x2/100)', 'intercepto': gamma0, 'coef_x1': gamma1,
     'coef_x2_o_z2': gamma2, 'riesgo_empirico': R_gamma, 'max_i d_i': max_d},
])
tabla_3

,modelo,intercepto,coef_x1,coef_x2_o_z2,riesgo_empirico,max_i d_i
0,beta (x2 original),-0.10147,5.012066,0.050477,0.923225,5.384582e-15
1,gamma (z2 = x2/100),-0.10147,5.012066,5.047693,0.923225,5.384582e-15
